# E4 — extend to 5 seeds (hardening, pre-registered TOST)

E4 (BIO span head + CLS head, jointly trained, `run_17_bio_cls_joint.py`) already has 3 seeds (42, 123, 7) registered: Joint F1 0.745±0.007, between System D (0.749) and E (0.736). This run adds 2 more seeds (**1, 99**) per the 2026-06-23 council hardening list — buys down the flat-reject tail, does NOT move venue tier on its own.

**TOST pre-registered BEFORE this run** — see `experiments/rigor/PREREG_E4_5seed_TOST.md`: margin ±0.02, bootstrap over per-example gaps pooled across all 5 seeds, alpha 0.05, against System D and System E separately. Do not change the margin/mode after seeing these 2 seeds' numbers.

**Persistence:** outputs symlinked to Drive by the runner; hard-fails before training if not Drive-backed. If the session times out, **just re-run the training cell** — finished seeds skip, resumes at the first incomplete one.

Run cells top to bottom. Use a **GPU** runtime (Runtime → Change runtime type → T4/A100).

In [ ]:
# 1. Config
REPO_URL = 'https://github.com/JustLetMeBeHello/Idiomator_Research.git'
BRANCH   = 'main'
REPO     = '/content/Idiomator_Research'   # absolute — never use a relative %cd
SEEDS    = '1 99'                            # the 2 NEW seeds only — 42/123/7 already done+registered
FORCE    = '0'                               # '1' retrains even completed seeds — never use to "make sure"
DRIVE_OUT = '/content/drive/MyDrive/IdiomatorRigor'
print('repo:', REPO, '| seeds:', SEEDS, '| force:', FORCE)

In [ ]:
# 2. Clone / refresh repo, pin absolute cwd, kill any nested duplicate clone
import os, subprocess, sys
from pathlib import Path
nested = os.path.join(REPO, 'Idiomator_Research')
if os.path.isdir(nested):
    subprocess.run(['rm', '-rf', nested], check=True)
if os.path.isdir(os.path.join(REPO, '.git')):
    subprocess.run(['git', '-C', REPO, 'fetch', '--quiet', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'checkout', '--quiet', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO, 'reset', '--hard', f'origin/{BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--quiet', '--branch', BRANCH, REPO_URL, REPO], check=True)
os.chdir(REPO)   # ABSOLUTE — the runner self-cds to git toplevel from here
print('cwd:', os.getcwd())
print('head:', subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
assert Path('experiments/rigor/run_17b_bio_cls_joint.sh').exists(), 'runner missing — wrong repo/branch?'
assert Path('experiments/rigor/PREREG_E4_5seed_TOST.md').exists(), 'pre-registration file missing — push it before running'

In [ ]:
# 3. Install deps + confirm GPU
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Requirements.txt'], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'NO GPU — switch runtime to T4/A100'

In [ ]:
# 4. Mount Drive + export env the runner reads
from google.colab import drive
drive.mount('/content/drive')
Path(DRIVE_OUT).mkdir(parents=True, exist_ok=True)
os.environ['DRIVE_OUT'] = DRIVE_OUT
os.environ['SEEDS']     = SEEDS
os.environ['FORCE']     = FORCE
print('DRIVE_OUT =', DRIVE_OUT)

In [ ]:
# 5. GPU smoke test (~1-2 min): 1 epoch, English only, throwaway dir
!python experiments/rigor/run_17_bio_cls_joint.py \
    --output_dir /tmp/_e4_smoke --langs English --test_langs English \
    --epochs 1 --batch_size 8

In [ ]:
# 6. Dry-run the real runner: prints the plan + persistence gate, trains nothing
!bash experiments/rigor/run_17b_bio_cls_joint.sh --dry-run

In [ ]:
# 7. FULL RUN — seeds 1 and 99 only, Drive-gated, resumable.
#    Re-run this cell after any timeout: finished seeds skip, resumes the rest.
LOG = f'{DRIVE_OUT}/e4_5seed_extend_console.log'
!bash experiments/rigor/run_17b_bio_cls_joint.sh 2>&1 | tee -a "$LOG"
print('\nlog →', LOG)

In [ ]:
# 8. Persistence readback for the 2 new seeds — reads metrics.json back FROM DRIVE
import json
EPOCH_CAP = 7
print(f"{'seed':<6}{'test_cls_f1':<14}{'test_exact':<12}{'test_ovlp_f1':<14}{'best_epoch':<12}{'best_dev_joint_f1':<18}{'persisted?'}")
for seed in SEEDS.split():
    mp = Path(DRIVE_OUT) / f'bio_cls_joint_mbert_s{seed}' / 'metrics.json'
    if not mp.exists():
        print(f'{seed:<6}— metrics.json NOT on Drive (incomplete / not persisted)')
        continue
    M = json.load(open(mp))
    flag = '  ⚠ at cap — check convergence' if M['best_epoch'] >= EPOCH_CAP else ''
    print(f"{seed:<6}{M['test_cls_macro_f1']:<14}{M['test_span_exact']:<12}{M['test_span_overlap']:<14}{M['best_epoch']:<12}{M['best_dev_joint_f1']:<18}yes{flag}")
print('\nNext (local, not in this notebook): pull both seeds, register via')
print('Full_evaluation.py --joint_preds, check_metric_drift.py, then re-run N1 TOST')
print('against System D/E using the full 5-seed pool per PREREG_E4_5seed_TOST.md.')